Hver fil må dekodes i ISO-8859-1/LATIN1 eller UTF-8, kanskje kan det sjekkes ved at hvis dekodingen returnerer første tegn som er riktig returnerer funksjonen True, og resten av filen leses med gitt dekoding. Hvis ikke så testes den andre dekodingen

Filtrer ut støy ved å si at alle linjer som ikke begynner med '[' er støy.
Videre kan hver linje legges til i ei liste hvor elementene per linje er adskilt med 'mellomrom'


Basert på oppgaveteksten så virker det som priser i EUR og USD er korrekte, med at kjøp gjennomført i NOK må konverteres fra EUR til NOK da verdien egentlig står i EUR.

TASK 1

In [50]:

from pathlib import Path
import json
import numpy as np
import math as m

In [40]:

def find_encoding(file: Path):
    """Checks if encoding of file is UTF-8, if not, sets encoding to Latin 1"""
    file = Path(file)
    try:
        with file.open(encoding= 'utf-8') as f:
            f.read()
        Etype = 'utf-8'
    
    except UnicodeDecodeError:
        Etype = 'iso-8859-1'

    return Etype

        
# I had to alter one of the elements in the 'Cgenerate_data_d2.py' because the vreated files only contained ASCII chars,
#so it wasnt possible to separate utf-8 from latin 1 without æ,ø,å. To solve, 'muffin' was changed to 'møffin'

a =find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/library_cafe/POS_01/register_sales.log')
b=find_encoding('2026/library_cafe/POS_02/register_sales.log')
c=find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/library_cafe/POS_03/register_sales.log')
d=find_encoding('C:/Users/danie/Documents/NMBU 4.år/Inf 201/Deliverables/W43_44/INF201_Deliverable_2/2026/science_building/POS_02/register_sales.log')
print(a, b, c, d)



utf-8 iso-8859-1 utf-8 utf-8


Write a log parser function that uses regular expressions. Log files contain valid sales
lines mixed with system noise.
Valid line format:
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
Noise lines to ignore:
--- PRINTER PAPER LOW WARNING ---
=== NETWORK SYNC OK ===
Construct a regular expression ( re module) to capture the 6 key fields from
valid lines:
timestamp (string, format: YYYY-MM-DD HH:MM:SS )
register_id (string, e.g., POS_01 )
item (string, e.g., Latte )
price (float, e.g., 4.50 )
currency (string, e.g., EUR , NOK , USD )
status (string: SUCCESS , REFUND , or CANCELLED )
Return a list of Python dictionaries, where each dictionary represents one valid
transaction record.

In [41]:
valid_lines = []
valid_transactions = []

def log_parser(file: Path)->list[dict]:
    """ Takes registerlogs as input, returns a list where each element is a dictionary of successfull transactions. """
   # opening file with correct input
    with open(file, encoding=find_encoding(file)) as f:
        line = f.readlines() #readlines create a list,line where each line in f becomes a object in that list.

    #step 1: removing all noise lines.
    for i in line:
        if i[0:2] =='[2':
            valid_lines.append(i) #if the object of line begins with [2, the first to chars of the time stamp of valid lines
    
    #step 2: Creating dictionaries for all valid lines, append them to a list of dictionaries    
    for valid_line in valid_lines:
        elements = valid_line.split()
        
        transaction = {
            'timestamp': elements[0].strip('[') +' '+ elements[1].strip(']'),
            'register' : elements[2].strip('[]'), 
            'item' : elements[3].split('=')[1],
            'price' : float(elements[4].split('=')[1]),
            'currency' : elements[5].strip(),
            'status' : elements[6].strip('()')
        }
        if transaction['status'] == 'SUCCESS':
            valid_transactions.append(transaction)
    
    return valid_transactions

log_parser('2026/library_cafe/POS_01/register_sales.log')



[{'timestamp': '2026-10-15 07:01:04',
  'register': 'POS_01',
  'item': 'Croissant',
  'price': 3.81,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:02:11',
  'register': 'POS_01',
  'item': 'Espresso',
  'price': 3.5,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:03:54',
  'register': 'POS_01',
  'item': 'Møffin',
  'price': 3.23,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:32',
  'register': 'POS_01',
  'item': 'Espresso',
  'price': 2.67,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:08:19',
  'register': 'POS_01',
  'item': 'Sandwich',
  'price': 7.59,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:08:44',
  'register': 'POS_01',
  'item': 'Sandwich',
  'price': 7.91,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:34',
  'register': 'POS_01',
  'item': 'Latte',
  'price': 4.49,
  'currency': 'EUR',
  'stat

Recap task 1: 
We see that using only Path and regular expressions we've created a set of functions that recieve raw log data from cashregisters, filters out noise lines and then sorts out only the successfull transactions. These transactions are then (as individual dictionaries) added to a list. The filtering for sorting out the lines that begin with '[2' and '[D', is maybe a bit primitive, but it will work until th timstamp hits th year 3000.

TASK 2

1. Write a recursive directory search function that finds all log files in a nested directory
and returns a list of pathlib.Path objects:
Convert the input path to a pathlib.Path object.
Iterate through items using Path.iterdir() .
If an item is a directory, recursively search inside it.
If an item is a file with a matching extension ( .log ), add its Path object to a
list.
Constraint: Do NOT use Path.glob() , Path.rglob() , or os.walk() . You
must implement explicit recursion.
2. Metadata about the different stores can be found in the
data/store_config.json file. This file contains deeply nested dictionaries and
lists (see example below). Write a recursive function that:
Traverses any combination of nested dictionaries and lists found in this file.
Returns a set (type hint: set[str] ) of all register id strings where "active":
true 

In [42]:
log_files = []


def dir_search(dirpath: str) -> list[str]:
    """Search through a directory. Add all paths of '.log' - files to a list. """
    dirpath = Path(dirpath)
    # 1. searching through input directory
    for file in dirpath.iterdir():
        if file.is_file() and file.suffix == '.log':
            log_files.append(file)
      
        elif file.is_dir():
            dir_search(file)
        
    return log_files
"""

def find_active(file: str) -> set[str]:
    """"""Returns at set of strings containing register name and whether it is active or not""""""
    active_ids = set()
    with open(file, 'r') as f:
        data = json.load(f)

    if isinstance(obj, dict):
        if obj.get('active') is True and 'id' in obj:
            active_ids.add(obj['id'])

        for value in obj.values():
            active_ids |= find_active(value)

    elif isinstance(obj, list):
        for item in obj:
            active_ids |= find_active(item)

    return active_ids


dir_search('2026')
find_active('store_config.json')

log_files
type(active_registers)
"""
dir_search('2026')

[WindowsPath('2026/library_cafe/POS_01/register_sales.log'),
 WindowsPath('2026/library_cafe/POS_02/register_sales.log'),
 WindowsPath('2026/library_cafe/POS_03/register_sales.log'),
 WindowsPath('2026/science_building/POS_01/register_sales.log'),
 WindowsPath('2026/science_building/POS_02/register_sales.log'),
 WindowsPath('2026/science_building/POS_03/register_sales.log'),
 WindowsPath('2026/student_union/POS_01/register_sales.log'),
 WindowsPath('2026/student_union/POS_02/register_sales.log'),
 WindowsPath('2026/student_union/POS_03/register_sales.log')]

TASK 3

 Write a function for removing price outliers that:
Converts input price data to a 1D NumPy float array.
Detects hardware glitch outliers ( ) where a value deviates from the mean
by more than 3 standard deviations:
Replaces detected outlier values with the median of the valid prices.
Returns the cleaned 1D array.
2. Write a function compute_refund_ratio(sales_count: np.ndarray,
refund_count: np.ndarray) -> np.ndarray that:
Computes the element-wise refund ratio using NumPy array operations:
The addition of prevents zero-division errors.
Task 4: Data analysis using Panda

In [43]:
price_list = []


def all_prices(dir):
    """Takes a directory as input, returns prices from all transactions"""
    filepaths = dir_search(dir)
    print(filepaths)
    for file in filepaths:
        price_list.append(log_parser(file))

        
all_prices('2026')


[WindowsPath('2026/library_cafe/POS_01/register_sales.log'), WindowsPath('2026/library_cafe/POS_02/register_sales.log'), WindowsPath('2026/library_cafe/POS_03/register_sales.log'), WindowsPath('2026/science_building/POS_01/register_sales.log'), WindowsPath('2026/science_building/POS_02/register_sales.log'), WindowsPath('2026/science_building/POS_03/register_sales.log'), WindowsPath('2026/student_union/POS_01/register_sales.log'), WindowsPath('2026/student_union/POS_02/register_sales.log'), WindowsPath('2026/student_union/POS_03/register_sales.log'), WindowsPath('2026/library_cafe/POS_01/register_sales.log'), WindowsPath('2026/library_cafe/POS_02/register_sales.log'), WindowsPath('2026/library_cafe/POS_03/register_sales.log'), WindowsPath('2026/science_building/POS_01/register_sales.log'), WindowsPath('2026/science_building/POS_02/register_sales.log'), WindowsPath('2026/science_building/POS_03/register_sales.log'), WindowsPath('2026/student_union/POS_01/register_sales.log'), WindowsPath

In [44]:
print(len(price_list))

18


In [ ]:

test_array = np.array([5,9,65,7,2,3,6,4,89,2,6,6,7,8,999,999,999,5,2,2,1,1,1,3,5,4,7,9,8,6,56,3,2,6])

def mean(array: np.array) -> float:
    """Returns mean value of numpy array"""
    return float(sum(array)/len(array))


def stdev(array: np.array) -> float:
    
    """Returns standard deviation of array"""
    for i in range(0, len(array)):
        var = (sum(i - mean(array)))/len(array)
    return m.sqrt(var)

a = mean(test_array)
print(a, type(a))
b = stdev(test_array)



98.1470588235294 <class 'float'>


TypeError: 'float' object is not iterable